In [ ]:
from pathlib import Path

from uavdiff.config import CFG
from uavdiff.agent_diffusion import AgentDiffusion
from uavdiff.experiment_utils import (create_experiment_dir, save_config_json, build_experiment_info, save_reward_and_buffer_plots, save_loss_plots)
PROJECT_ROOT = Path.cwd().parent
LOG_ROOT = PROJECT_ROOT / "logs"
LOG_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
EXP_NAME = "uav3_1_1"

USER_NOTES = """
This experiment tests the diffusion policy under the forced-return setting.
The UAVs are forced to return to the airship center every 10 executed steps.
The goal is to make the policy learn more trajectories that start from the center,
which is more consistent with the upper-level dispatch / charging logic.
""".strip()

In [ ]:
cfg = CFG

# -----------------------------
# Environment
# -----------------------------
cfg.env.n_uavs = 3
cfg.env.world_size = 400.0
cfg.env.cov_radius = 86.6
cfg.env.lam = 0.05

cfg.env.ratio_reward = True
cfg.env.ratio_weight = 10.0
cfg.env.team_max_sum_weight = 0.3
cfg.env.max_before_weight = 1.0
cfg.env.return_to_center_every = 10

# -----------------------------
# Observation
# -----------------------------
cfg.obs.mode = "lv+uav+topk"
cfg.obs.topk_oldest = 1

# -----------------------------
# Chunk / execution
# -----------------------------
cfg.chunk.chunk_len = 1
cfg.chunk.exec_len = 1

# -----------------------------
# Diffusion
# -----------------------------
cfg.diffusion.predict_type = "eps"
cfg.diffusion.train_diffusion_steps = 32
cfg.diffusion.hidden_dim = 256
cfg.diffusion.n_blocks = 4
cfg.diffusion.n_heads = 4
cfg.diffusion.bc_weight = 1.0
cfg.diffusion.q_guidance_weight = 0.1

# -----------------------------
# Critic
# -----------------------------
cfg.critic.state_horizon = 1
cfg.critic.gamma = 0.99
cfg.critic.target_tau = 0.005

# -----------------------------
# Replay
# -----------------------------
cfg.replay.capacity = 100_000
cfg.replay.batch_size_actor = 128
cfg.replay.batch_size_critic = 128
cfg.replay.actor_chunk_len = cfg.chunk.chunk_len

# -----------------------------
# Training
# -----------------------------
cfg.train.num_episodes = 200
cfg.train.num_timestep = 100
cfg.train.warmup_steps = 1000
cfg.train.lr_actor = 3e-4
cfg.train.lr_critic = 3e-4
cfg.train.eval_every = 5

cfg.validate()
print("Config ready.")

In [ ]:
EXP_INFO = build_experiment_info(cfg, EXP_NAME, USER_NOTES)
print(EXP_INFO)

In [ ]:
exp_dir = create_experiment_dir(root=str(LOG_ROOT), exp_name=EXP_NAME, exp_info=EXP_INFO, allow_overwrite=True)
save_config_json(cfg, exp_dir / "config.json")
print("Experiment directory:", exp_dir)

In [ ]:
agent = AgentDiffusion(cfg)
print("Agent initialized.")
print("obs_dim =", agent.obs_dim)
print("act_dim =", agent.act_dim)

In [ ]:
agent.train(num_episodes=cfg.train.num_episodes, num_timestep=cfg.train.num_timestep, eval_every=cfg.train.eval_every, render_eval=True)

In [ ]:
save_reward_and_buffer_plots(agent, cfg, exp_dir)
save_loss_plots(agent, exp_dir)
print("Saved plots to:", exp_dir)

In [ ]:
final_ret, final_buf = agent.evaluate(num_timestep=cfg.train.num_timestep, deterministic=True,render=True,)
print("Final Eval Return =", final_ret)
print("Final Eval Avg MaxBuffer =", final_buf)

## Save model and training logs

In [ ]:
agent.save_training_logs(str(exp_dir / "train_logs.npz"))
agent.save(str(exp_dir / "model.pt"))
print("Saved:")
print("-", exp_dir / "config.json")
print("-", exp_dir / "exp_info.txt")
print("-", exp_dir / "train_logs.npz")
print("-", exp_dir / "model.pt")

## Summary

In [ ]:
summary = {
    "exp_name": EXP_NAME,
    "final_eval_return": float(final_ret),
    "final_eval_avg_maxbuffer": float(final_buf),
    "num_train_rewards": len(agent.train_rewards),
    "num_eval_rewards": len(agent.eval_rewards),
    "num_logged_critic_losses": len(agent.critic_loss_log),
}
summary